## 01.Foundation Model Inference with Amazon Bedrock

### 학습 목표
1. Amazon Bedrock 클라이언트를 초기화하고 사용 가능한 Foundation Model 목록을 조회한다.
2. `invoke_model` API로 Claude에게 첫 번째 프롬프트를 전송한다.
3. `converse` API(통합 인터페이스)로 멀티턴 대화를 구현한다.

### Amazon Bedrock 아키텍처
```
사용자 코드
   │
   ├─ bedrock          (모델 목록 조회)
   ├─ bedrock-runtime  (텍스트 생성, 스트리밍)
   └─ bedrock-agent-runtime  (RAG / Knowledge Base)
```

### 주요 모델 ID (2025 기준)
```
anthropic.claude-3-5-sonnet-20241022-v2:0  ← 최신 고성능
anthropic.claude-3-haiku-20240307-v1:0    ← 빠르고 경제적
amazon.titan-text-premier-v1:0            ← AWS 자체 모델
meta.llama3-8b-instruct-v1:0              ← 오픈소스 계열
```


In [ ]:
#환경 초기화
import boto3, json
import pandas as pd

# Bedrock 클라이언트 (모델 목록 조회용)
bedrock = boto3.client('bedrock', region_name='us-east-1')

# Bedrock Runtime 클라이언트 (텍스트 생성용)
bedrock_runtime = boto3.client('bedrock-runtime', region_name='us-east-1')

print('Bedrock 클라이언트 생성 완료')
print(f'  bedrock           : {bedrock.meta.service_model.service_name}')
print(f'  bedrock-runtime   : {bedrock_runtime.meta.service_model.service_name}')


In [ ]:
# list_foundation_models 

response = bedrock.list_foundation_models(
    byOutputModality='TEXT'
)

models = response['modelSummaries']
print(f'텍스트 생성 모델 수: {len(models)}개\n')

rows = []
for m in models:
    rows.append({
        '모델ID'    : m['modelId'], 
        '제공사'    : m['providerName'],
        '모델명'    : m['modelName'],
    })

df = pd.DataFrame(rows)
print(df[df['모델ID'].str.contains('claude|titan|llama')].to_string(index=False))


In [ ]:
# invoke_model API로 Claude에게 질문 전송하기
# us-east-1이면 프리픽스는 us.

MODEL_ID = 'us.anthropic.claude-sonnet-4-20250514-v1:0'

payload = {
    'anthropic_version': 'bedrock-2023-05-31',
    'max_tokens': 1024,
    'messages': [
        {
            'role': 'user',  
            'content': 'AWS Bedrock을 한 문장으로 설명해줘.'  
        }
    ]
}

response = bedrock_runtime.invoke_model(
    modelId=MODEL_ID, 
    body=json.dumps(payload), 
    contentType='application/json'
)

result = json.loads(response['body'].read())
answer = result['content'][0]['text']
print('Claude 응답:')
print(answer)


In [ ]:
# invoke_model API로 Nova에게 질문을 전송
MODEL_ID = 'amazon.nova-micro-v1:0'

payload = {
    'schemaVersion': 'messages-v1',    # Nova 전용 필드
    'inferenceConfig': {'maxTokens': 1024},  # 응답으로 생성할 최대 토큰 수 (정수)
    'messages': [
        {
            'role': 'user',             # 대화에서 사람 쪽 역할을 나타내는 문자열
            'content': [{'text': 'AWS Bedrock을 한 문장으로 설명해줘.'}]  # 모델에게 보낼 프롬프트 문자열 입력
        }
    ]
}

response = bedrock_runtime.invoke_model(
    modelId=MODEL_ID,                    
    body=json.dumps(payload),        
    contentType='application/json'
)

result = json.loads(response['body'].read())
answer = result['output']['message']['content'][0]['text']
print('Nova 응답:')
print(answer)

In [ ]:
# Temperature & Max Tokens 파라미터 실험
def call_claude(prompt, temperature=0.7, max_tokens=512):
    payload = {
        'anthropic_version': 'bedrock-2023-05-31',
        'max_tokens': max_tokens,
        'temperature': temperature,
        'messages': [{'role': 'user', 'content': prompt}]
    }
    resp = bedrock_runtime.invoke_model(
        modelId=MODEL_ID, body=json.dumps(payload), contentType='application/json'
    )
    return json.loads(resp['body'].read())['content'][0]['text']

prompt = '파이썬의 장점을 세 가지만 알려줘.'
for temp in [0.1, 0.7, 1.2]:
    print(f'\n--- temperature={temp} ---')
    print(call_claude(prompt, temperature=temp))


In [ ]:
# converse API로 멀티턴 대화를 구현 

conversation_history = []

def chat(user_message, system_prompt=None):
    """대화 히스토리를 유지하며 converse API를 호출합니다"""
    conversation_history.append({
        'role': 'user',
        'content': [{'text': user_message}]
    })

    kwargs = {
        'modelId': MODEL_ID,
        'messages': conversation_history,
        'inferenceConfig': {'maxTokens': 512, 'temperature': 0.7}
    }
    if system_prompt:
        kwargs['system'] = [{'text': system_prompt}]

    response = bedrock_runtime.converse(**kwargs)

    assistant_msg = response['output']['message']['content'][0]['text']
    conversation_history.append({
        'role': 'assistant',
        'content': [{'text': assistant_msg}]
    })
    return assistant_msg

# 멀티턴 대화 테스트
SYSTEM = '당신은 친절한 AI 어시스턴트입니다. 한국어로 답변하세요.'

print('사용자:', '안녕하세요! 클라우드 컴퓨팅이 뭔가요?')
print('AI:', chat('안녕하세요! 클라우드 컴퓨팅이 뭔가요?', SYSTEM))
print()
print('사용자:', 'AWS와 Azure 중 어떤 게 더 좋아요?')
print('AI:', chat('AWS와 Azure 중 어떤 게 더 좋아요?'))
print()
print(f'대화 히스토리 길이: {len(conversation_history)}개 메시지')
